## HMM Distillation Tutorial

Step 1. sampling data from the base model

In [4]:
BASE_MODEL_PATH = 'gpt2-large'

# a list of prompts used to sample data from the given LLM, 
# please change according to the LLM & your use cases.
INPUT_FILE = './workspace/inference_data/distillation_prompts.json' 

DATASET = 'gpt2-large'
DATA_PATH = f'./workspace/hmm_data/{DATASET}'
LVD_SIZE = 100000
CHUNK_SIZE = 100000
DEV_SIZE = 20000
TOTAL_CHUNKS = 100
SEQUENCE_LEN = 32

In [ ]:
import os


CUDA_CORES = '0,1,2,3,4,5'
BATCH_SIZE = 512


# create data path
os.system(f'mkdir -p {DATA_PATH}')


# sample LVD_SIZE examples for initializing hmm parameters via latent variable distillation (LVD)
cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} torchrun --standalone --nproc_per_node=gpu \
    sample_data.py \
    --model_name_or_path {BASE_MODEL_PATH} \
    --tokenizer_name_or_path {BASE_MODEL_PATH} \
    --input_file {INPUT_FILE} --chunk_size {LVD_SIZE} \
    --batch_size {BATCH_SIZE} --max_new_tokens {SEQUENCE_LEN} \
    --save_embeddings --output_file {DATA_PATH}/{DATASET}.lvd'.strip()
print(cmd)


# sample TOTAL_CHUNKS chunks of training examples as the training set
cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} torchrun --standalone --nproc_per_node=gpu \
    sample_data.py \
    --model_name_or_path {BASE_MODEL_PATH} \
    --tokenizer_name_or_path {BASE_MODEL_PATH} \
    --input_file {INPUT_FILE} --chunk_size {CHUNK_SIZE} --total_chunks {TOTAL_CHUNKS} \
    --batch_size {BATCH_SIZE} --max_new_tokens {SEQUENCE_LEN} \
    --output_file {DATA_PATH}/{DATASET}.train'.strip()
print(cmd)


# sample DEV_SIZE examples as the dev set
cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} torchrun --standalone --nproc_per_node=gpu \
    sample_data.py \
    --model_name_or_path {BASE_MODEL_PATH} \
    --tokenizer_name_or_path {BASE_MODEL_PATH} \
    --input_file {INPUT_FILE} --chunk_size {DEV_SIZE} \
    --batch_size {BATCH_SIZE} --max_new_tokens {SEQUENCE_LEN} \
    --output_file {DATA_PATH}/{DATASET}.dev'.strip()
print(cmd)

Step 2. initialize checkpoint-0 for training HMM via latent variable distillation (LVD)

In [5]:
import os
from transformers import AutoTokenizer

# specify the HMM size
HIDDEN_STATES = 4096

# get vocab_size and eos_token_id; might vary for different models #
__tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
VOCAB_SIZE = __tokenizer.vocab_size
EOS_TOKEN_ID = __tokenizer.eos_token_id
####################################################################

HMM_MODEL_ID = f'hmm_{DATASET}_{HIDDEN_STATES}'
HMM_MODEL_PATH = f'./workspace/models/{HMM_MODEL_ID}/'

_ = os.system(f'mkdir -p {HMM_MODEL_PATH}')

In [6]:
import os

CUDA_CORES = '0,1,2,3,4,5'
SEQUENCES_FILE = f'{DATA_PATH}/{DATASET}.lvd'
EMEBEDDINGS_FILE = f'{DATA_PATH}/{DATASET}.lvd.embeddings'

# latent variable distillation
cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} python lvd_hmm.py \
    --sequences_file {SEQUENCES_FILE} --embeddings_file {EMEBEDDINGS_FILE} \
    --hidden_states {HIDDEN_STATES} --vocab_size {VOCAB_SIZE} --eos_token_id {EOS_TOKEN_ID} \
    --kmeans_iterations 100 --pseudocount 0.001 \
    --output_file {HMM_MODEL_PATH}/checkpoint-0'
print(cmd)

CUDA_VISIBLE_DEVICES=0,1,2,3,4,5 python lvd_hmm.py     --sequences_file ./workspace/hmm_data/gpt2-large/gpt2-large.lvd --embeddings_file ./workspace/hmm_data/gpt2-large/gpt2-large.lvd.embeddings     --hidden_states 4096 --vocab_size 50257 --eos_token_id 50256     --kmeans_iterations 100 --pseudocount 0.001     --output_file ./workspace/models/hmm_gpt2-large_4096//checkpoint-0


In [8]:
! CUDA_VISIBLE_DEVICES=0,1,2,3,4,5 python lvd_hmm.py     --sequences_file ./workspace/hmm_data/gpt2-large/gpt2-large.lvd --embeddings_file ./workspace/hmm_data/gpt2-large/gpt2-large.lvd.embeddings     --hidden_states 4096 --vocab_size 50257 --eos_token_id 50256     --kmeans_iterations 100 --pseudocount 0.001     --output_file ./workspace/models/hmm_gpt2-large_4096//checkpoint-0


loading embeddings from ./workspace/hmm_data/gpt2-large/gpt2-large.lvd.embeddings ...
/home/s/sukumarg/uncertainty/Ctrl-G/distillation/lvd_hmm.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this e

Step 3. train HMM via Expectation Maximization (EM)

In [9]:
import os

os.system('mkdir -p ./workspace/logs')
LOG_FILE=f'./workspace/logs/{HMM_MODEL_ID}_log.txt'

CUDA_CORES = '0,1,2,3,4,5'
BATCH_SIZE = 256
SAVE_PER_STEP = 10
DROPOUT = 0.01

# EM training schedule:
# 1. train for 10 EM steps, each step using 1 chunk of data
# 2. train for 5 EM steps, each step using 2 chunks of data
# 3. train for 4 EM steps, each step using 5 chunks of data
# 4. train for 4 EM steps, each step using 10 chunks of data
# 5. train for 4 EM steps, each step using 20 chunks of data
# 6. train for 1 EM steps, each step using 40 chunks of data
EM_SCHEDULE = "\"10,1;5,2;4,5;4,10;4,20;1,40\""

cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} torchrun --standalone --nproc_per_node=gpu train_hmm.py \
    --model_path {HMM_MODEL_PATH} --checkpoint 0 --save_per_step {SAVE_PER_STEP} \
    --data_path {DATA_PATH} --dataset {DATASET} --total_chunks {TOTAL_CHUNKS} --batch_size {BATCH_SIZE} \
    --em_schedule {EM_SCHEDULE} --dropout {DROPOUT} --log_file {LOG_FILE}'.strip()
print(cmd)

CUDA_VISIBLE_DEVICES=0,1,2,3,4,5 torchrun --standalone --nproc_per_node=gpu train_hmm.py     --model_path ./workspace/models/hmm_gpt2-large_4096/ --checkpoint 0 --save_per_step 10     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100 --batch_size 256     --em_schedule "10,1;5,2;4,5;4,10;4,20;1,40" --dropout 0.01 --log_file ./workspace/logs/hmm_gpt2-large_4096_log.txt


In [10]:
!CUDA_VISIBLE_DEVICES=0,1,2,3,4,5 torchrun --standalone --nproc_per_node=gpu train_hmm.py     --model_path ./workspace/models/hmm_gpt2-large_4096/ --checkpoint 0 --save_per_step 10     --data_path ./workspace/hmm_data/gpt2-large --dataset gpt2-large --total_chunks 100 --batch_size 256     --em_schedule "10,1;5,2;4,5;4,10;4,20;1,40" --dropout 0.01 --log_file ./workspace/logs/hmm_gpt2-large_4096_log.txt

Loading weights from local directory
/home/s/sukumarg/uncertainty/Ctrl-G/distillation/train_hmm.py:31: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dev_data = torch.load(f'

In [12]:
!pip install -r requirements.txt

  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 256.7 kB/s  0:00:10 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 2.6 MB/s  0:01:19m0:00:0100:050m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 14.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 33.0 MB/s  0:00:00
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 43.7 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 20.1 MB/s  0:00:00 eta 0:00:01
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.7 MB/s 

In [13]:
!nvidia-smi

Mon Apr 13 01:13:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:98:00.0 Off |                    0 |
| N/A   47C    P0             68W /  300W |   65971MiB /  81920MiB |      1%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----